In [ ]:
# Cell 1.1: start the checkpoint download in the background, with a live progress bar
import os
import pathlib
import subprocess
import sys
import threading
import time

from IPython.display import HTML, display

T_START = time.perf_counter()
STAGES = {}  # stage name -> seconds since T_START; the summary at the end reads this

# The weights: Gemma 4 E2B's QAT 4-bit weights as one Q4_0 GGUF file (2.62 GB), loaded onto the chip
# directly. The safetensors repo supplies only config.json, the tokenizer and the chat template.
GGUF_REPO = "xbill9/gemma-4-E2B-it-qat-q4_0-exact-gguf"
GGUF_FILE = "gemma-4-E2B-it-q4_0-exact.gguf"
GGUF_REV = "74b2c92c8f7f84d3d9c39e8026f6a28f62788255"
MODEL_ID = "xbill9/gemma-4-E2B-it-qat-q4_0-w4a16-ct-text"
MODEL_REV = "315aec06cff196fd6479aa2669bf9379b0d0268a"
WORK = pathlib.Path("/content" if os.path.isdir("/content") else os.path.expanduser("~/gemma4-colab"))
WORK.mkdir(parents=True, exist_ok=True)

from huggingface_hub import HfApi, snapshot_download

# The small files first, in this cell (about a second): the engine compiles from config.json while
# the weights download.
MODEL_DIR = pathlib.Path(
    snapshot_download(MODEL_ID, revision=MODEL_REV, ignore_patterns=["*.safetensors"], token=False)
)
gguf_meta = HfApi(token=False).model_info(GGUF_REPO, revision=GGUF_REV, files_metadata=True).siblings
TOTAL_BYTES = next(f.size for f in gguf_meta if f.rfilename == GGUF_FILE)  # public: no token
print(
    f"config and tokenizer: {MODEL_ID} @ {MODEL_REV[:8]}, {sum(f.stat().st_size for f in MODEL_DIR.iterdir()) / 1e6:.0f} MB"
)
print(f"weights:              {GGUF_REPO} @ {GGUF_REV[:8]}, {GGUF_FILE}, {TOTAL_BYTES / 1e9:.2f} GB")
print("estimate: about 25 s. On a Colab v5e-1 on 2026-11-20 this file downloaded in 25.0 s.")
print("          The bar below measures this runtime's own rate.")
print("          The next cells run while it downloads; there is no need to wait here.\n", flush=True)

DOWNLOAD = """
import sys
from huggingface_hub import hf_hub_download
print(hf_hub_download(sys.argv[1], sys.argv[2], revision=sys.argv[3], token=False))
"""
download_log = open(WORK / "download.log", "w")
download = subprocess.Popen(
    [sys.executable, "-c", DOWNLOAD, GGUF_REPO, GGUF_FILE, GGUF_REV],
    stdout=subprocess.PIPE,
    stderr=download_log,
    text=True,
)

# The engine's code (tpu-jax, pinned) and the OpenAI client install in the background too.
import importlib.util  # noqa: E402

TPU_JAX_COMMIT = "4b9f8e94619f1995206617e07e3c4ac62685d47b"
ENGINE_DIR = WORK / "tpu-jax"
prep_steps = []
if not (ENGINE_DIR / ".git").exists():
    prep_steps.append(f"git clone -q https://github.com/xbill9/tpu-jax {ENGINE_DIR}")
prep_steps.append(f"git -C {ENGINE_DIR} checkout -q {TPU_JAX_COMMIT}")
missing = [p for p in ("safetensors", "tokenizers") if importlib.util.find_spec(p) is None]
if missing:
    prep_steps.append(f"{sys.executable} -m pip install -q {' '.join(missing)}")
prep_log = open(WORK / "prep.log", "w")
# The OpenAI client is first used in section 3, so its install runs on its own and nothing waits for it.
OPENAI_INSTALL = None
if importlib.util.find_spec("openai") is None:
    OPENAI_INSTALL = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", "-q", "openai"], stdout=prep_log, stderr=prep_log
    )
PREP = subprocess.Popen(["bash", "-c", "set -e; " + "; ".join(prep_steps)], stdout=prep_log, stderr=prep_log)


def bytes_received():
    """Bytes this VM has received on its network interfaces, from /proc/net/dev."""
    total = 0
    for line in open("/proc/net/dev").read().splitlines()[2:]:
        name, data = line.split(":", 1)
        if name.strip() != "lo":
            total += int(data.split()[0])
    return total


def bar(frac, text):
    return HTML(
        f'<progress value="{frac * 100:.1f}" max="100" style="width:55%;height:18px"></progress>'
        f'&nbsp;<code style="font-size:13px">{text}</code>'
    )


DL = {"updates": 0}  # filled by the watcher: "path", "seconds", "error", and how often it redrew the bar


def watch_download(handle):
    """Update the bar once a second from network bytes, then collect the download's result.

    Progress is read from bytes received rather than from files in the cache, because the
    downloader may write each file only when it is complete. Other traffic in the meantime
    (pip, git) counts too, so the bar is capped at 99% until the download process exits.
    """
    rx0, t0 = bytes_received(), time.perf_counter()
    while download.poll() is None:
        got, el = bytes_received() - rx0, time.perf_counter() - t0
        rate = got / el if el > 0 else 0
        if rate < 1e6:
            left = "measuring the rate"
        elif got < TOTAL_BYTES:
            left = f"about {(TOTAL_BYTES - got) / rate:.0f} s left"
        else:
            left = "finishing"
        handle.update(bar(min(got / TOTAL_BYTES, 0.99),
                          f"{got / 1e9:5.2f} of {TOTAL_BYTES / 1e9:.2f} GB  {rate / 1e6:5.0f} MB/s  {left}"))  # fmt: skip
        DL["updates"] += 1
        time.sleep(1)
    DL["seconds"] = time.perf_counter() - t0
    DL["done_at"] = time.perf_counter() - T_START
    out = download.stdout.read()
    download_log.close()
    if download.returncode == 0:
        DL["path"] = pathlib.Path(out.strip().splitlines()[-1])
        handle.update(bar(1.0, f"downloaded {TOTAL_BYTES / 1e9:.2f} GB in {DL['seconds']:.0f} s"))
    else:
        DL["error"] = (WORK / "download.log").read_text()[-2000:]
        handle.update(bar(0, "download failed: see download.log"))


progress = display(bar(0, "starting"), display_id=True)
DOWNLOAD_THREAD = threading.Thread(target=watch_download, args=(progress,), daemon=True)
DOWNLOAD_THREAD.start()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

config and tokenizer: xbill9/gemma-4-E2B-it-qat-q4_0-w4a16-ct-text @ 315aec06, 32 MB
weights:              xbill9/gemma-4-E2B-it-qat-q4_0-exact-gguf @ 74b2c92c, gemma-4-E2B-it-q4_0-exact.gguf, 2.62 GB
estimate: about 25 s. On a Colab v5e-1 on 2026-11-20 this file downloaded in 25.0 s.
          The bar below measures this runtime's own rate.
          The next cells run while it downloads; there is no need to wait here.



In [ ]:
# Cell 1.2: the chip, the host and the software versions
import importlib.metadata as md
import platform
import shutil

import jax
import jax.numpy as jnp


def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return "not installed"


devices = jax.devices()
dev = devices[0]
if dev.platform != "tpu":
    raise RuntimeError("This notebook needs a TPU runtime: Runtime > Change runtime type > v5e-1 TPU")

mem = dev.memory_stats()
HBM_LIMIT = mem["bytes_limit"]
host_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
cpu_model = next(
    (line.split(":", 1)[1].strip() for line in open("/proc/cpuinfo") if line.startswith("model name")), "unknown"
)

print("software")
print(f"  Python        {platform.python_version()}")
for pkg in ("jax", "jaxlib", "libtpu", "transformers", "huggingface_hub"):
    print(f"  {pkg:13s} {version(pkg)}")
print("\nTPU")
print(f"  devices       {len(devices)} x {dev.device_kind}")
print(f"  HBM           {HBM_LIMIT / 1e9:.2f} GB, {mem['bytes_in_use'] / 1e9:.2f} GB in use")
for attr in ("coords", "core_on_chip"):
    if hasattr(dev, attr):
        print(f"  {attr:13s} {getattr(dev, attr)}")
for key in sorted(k for k in os.environ if k.startswith(("TPU_", "COLAB_TPU"))):
    print(f"  ${key} = {os.environ[key]}")
print("\nhost")
print(f"  CPU           {os.cpu_count()} x {cpu_model}")
print(f"  RAM           {host_ram / 1e9:.1f} GB")
print(f"  free disk     {shutil.disk_usage(WORK).free / 1e9:.0f} GB")
STAGES["chip info"] = time.perf_counter() - T_START

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


software
  Python        3.13.18
  jax           0.11.2
  jaxlib        0.11.2
  libtpu        0.0.50
  transformers  5.22.1
  huggingface_hub 1.33.0

TPU
  devices       1 x TPU v5 lite
  HBM           16.91 GB, 0.00 GB in use
  coords        [0, 0, 0]
  core_on_chip  0
  $TPU_ACCELERATOR_TYPE = v5e-1
  $TPU_CHIPS_PER_HOST_BOUNDS = 2,2,1
  $TPU_HOST_BOUNDS = 1,1,1
  $TPU_LIBRARY_PATH = /usr/local/lib/python3.13/dist-packages/libtpu/libtpu.so
  $TPU_ML_PLATFORM = JAX
  $TPU_ML_PLATFORM_VERSION = 0.11.2
  $TPU_RUNTIME_METRICS_PORTS = 8431,8432,8433,8434
  $TPU_SKIP_MDS_QUERY = 1
  $TPU_WORKER_HOSTNAMES = localhost
  $TPU_WORKER_ID = 0

host
  CPU           24 x AMD EPYC 7B13
  RAM           50.5 GB
  free disk     240 GB


In [ ]:
# Cell 2.1: the engine, pinned, and the checkpoint on the chip
import json

if PREP.wait() != 0:
    raise RuntimeError(f"engine setup failed:\n{(WORK / 'prep.log').read_text()[-2000:]}")
sys.path.insert(0, str(ENGINE_DIR))
from jax_engine import JaxGemmaEngine  # noqa: E402


import io  # noqa: E402
import re  # noqa: E402
import struct  # noqa: E402
import urllib.request  # noqa: E402

import numpy as np  # noqa: E402
import safetensors.flax  # noqa: E402
from ports.gemma4.jax_e_loader import convert_safetensors_to_jax_params  # noqa: E402

import datetime  # noqa: E402

import jinja2  # noqa: E402
import jinja2.ext  # noqa: E402
import jinja2.sandbox  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402


# The tokenizer and chat template come straight from `tokenizers` and Jinja: transformers would import
# PyTorch first, which took about 10 s here. On 2026-11-20 this matched transformers' AutoTokenizer on
# every test conversation (16 of 16, thinking on and off) and on 600 of 600 random decodes.
class ChatTokenizer:
    """The checkpoint's tokenizer.json and chat_template.jinja, rendered the way transformers renders
    them (a sandboxed Jinja environment with trim_blocks, lstrip_blocks and loop controls)."""

    def __init__(self, model_dir):
        self.tk = Tokenizer.from_file(str(model_dir / "tokenizer.json"))
        cfg = json.loads((model_dir / "tokenizer_config.json").read_text())
        tok = lambda t: t["content"] if isinstance(t, dict) else t  # noqa: E731
        self.bos_token, self.eos_token = tok(cfg.get("bos_token")), tok(cfg.get("eos_token"))
        self.bos_token_id = self.tk.token_to_id(self.bos_token)
        env = jinja2.sandbox.ImmutableSandboxedEnvironment(
            trim_blocks=True, lstrip_blocks=True, extensions=[jinja2.ext.loopcontrols]
        )

        def raise_exception(message):
            raise jinja2.exceptions.TemplateError(message)

        env.filters["tojson"] = lambda x, ensure_ascii=False, indent=None, separators=None, sort_keys=False: json.dumps(
            x, ensure_ascii=ensure_ascii, indent=indent, separators=separators, sort_keys=sort_keys
        )
        env.globals["raise_exception"] = raise_exception
        env.globals["strftime_now"] = lambda fmt: datetime.datetime.now().strftime(fmt)
        self.template = env.from_string((model_dir / "chat_template.jinja").read_text())

    def apply_chat_template(self, messages, tokenize=True, add_generation_prompt=False, **kwargs):
        text = self.template.render(
            messages=messages,
            add_generation_prompt=add_generation_prompt,
            bos_token=self.bos_token,
            eos_token=self.eos_token,
            **kwargs,
        )
        return self.encode(text) if tokenize else text

    def encode(self, text, add_special_tokens=False):
        return self.tk.encode(text, add_special_tokens=add_special_tokens).ids

    def decode(self, ids, skip_special_tokens=False):
        return self.tk.decode(list(ids), skip_special_tokens=skip_special_tokens)

    def convert_tokens_to_ids(self, token):
        return self.tk.token_to_id(token)


# --- The GGUF, read as the tensors tpu-jax reads from safetensors ---------------------------------
# GGUF names -> the safetensors names the engine expects. rope_freqs is skipped: the engine computes
# RoPE from config.json.
GGUF_TOP = {
    "token_embd.weight": "embed_tokens.weight",
    "per_layer_token_embd.weight": "embed_tokens_per_layer.weight",
    "per_layer_model_proj.weight": "per_layer_model_projection.weight",
    "per_layer_proj_norm.weight": "per_layer_projection_norm.weight",
    "output_norm.weight": "norm.weight",
}
GGUF_BLK = {
    "attn_q": "self_attn.q_proj.weight", "attn_k": "self_attn.k_proj.weight",
    "attn_v": "self_attn.v_proj.weight", "attn_output": "self_attn.o_proj.weight",
    "attn_q_norm": "self_attn.q_norm.weight", "attn_k_norm": "self_attn.k_norm.weight",
    "ffn_gate": "mlp.gate_proj.weight", "ffn_up": "mlp.up_proj.weight", "ffn_down": "mlp.down_proj.weight",
    "inp_gate": "per_layer_input_gate.weight", "proj": "per_layer_projection.weight",
    "attn_norm": "input_layernorm.weight", "post_attention_norm": "post_attention_layernorm.weight",
    "ffn_norm": "pre_feedforward_layernorm.weight", "post_ffw_norm": "post_feedforward_layernorm.weight",
    "post_norm": "post_per_layer_input_norm.weight", "layer_output_scale": "layer_scalar",
}  # fmt: skip
GGUF_F32, GGUF_Q4_0 = 0, 2


def safetensors_name(gguf_name):
    if gguf_name in GGUF_TOP:
        return "model.language_model." + GGUF_TOP[gguf_name]
    m = re.fullmatch(r"blk\.(\d+)\.(.+)\.weight", gguf_name)
    if not m or m.group(2) not in GGUF_BLK:
        raise KeyError(f"GGUF tensor {gguf_name} has no tpu-jax counterpart")
    return f"model.language_model.layers.{m.group(1)}.{GGUF_BLK[m.group(2)]}"


def gguf_index(f):
    """GGUF v3 header -> (offset of the tensor data, [(safetensors name, [out, in] shape, type, offset)])."""
    r = lambda fmt: struct.unpack("<" + fmt, f.read(struct.calcsize("<" + fmt)))  # noqa: E731
    if f.read(4) != b"GGUF":
        raise ValueError("not a GGUF file")
    r("I")
    n_tensors, n_kv = r("QQ")
    align, fixed = 32, {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}
    fmt = {0: "B", 1: "b", 2: "H", 3: "h", 4: "I", 5: "i", 6: "f", 7: "?", 10: "Q", 11: "q", 12: "d"}

    def value(t):  # metadata values are skipped, except the data alignment
        if t == 8:
            return f.read(r("Q")[0])
        if t == 9:
            at, n = r("IQ")
            if at in fixed:
                return f.seek(fixed[at] * n, 1)
            return [value(at) for _ in range(n)]
        return r(fmt[t])[0]

    for _ in range(n_kv):
        key = f.read(r("Q")[0]).decode()
        v = value(r("I")[0])
        if key == "general.alignment":
            align = v
    index = []
    for _ in range(n_tensors):
        name = f.read(r("Q")[0]).decode()
        dims = r("Q" * r("I")[0])
        t, off = r("IQ")
        if name != "rope_freqs.weight":
            index.append((safetensors_name(name), tuple(int(d) for d in reversed(dims)), t, off))
    p = f.tell()
    return p + (-p % align), index


def q4_0_expand(blocks):
    """uint8 [rows, n, 18] ggml block_q4_0 -> bf16 [rows, n*32]. A block is an fp16 step d and 16 bytes:
    low nibbles are elements 0..15, high nibbles 16..31, each q + 8. The value is bf16(q * d), the
    single rounding the engine's own 4-bit unpack gives."""
    d = jax.lax.bitcast_convert_type(
        blocks[..., 0].astype(jnp.uint16) | (blocks[..., 1].astype(jnp.uint16) << 8), jnp.float16
    )
    qs = blocks[..., 2:]
    q = jnp.concatenate([qs & 0xF, qs >> 4], axis=-1).astype(jnp.int32) - 8
    v = (q.astype(jnp.float32) * d.astype(jnp.float32)[..., None]).astype(jnp.bfloat16)
    return v.reshape(blocks.shape[0], -1)


q4_0_expand_jit = jax.jit(q4_0_expand)
q4_0_expand_into = jax.jit(
    lambda out, blocks, row: jax.lax.dynamic_update_slice(out, q4_0_expand(blocks), (row, 0)), donate_argnums=0
)
CHUNK_ROWS = 16384  # the two embedding tables expand in chunks, so neither needs an f32 copy


def gguf_to_chip(path, data_start, index):
    """Only the 4-bit blocks cross to the chip (2.6 GB); each tensor is expanded to bf16 there."""
    mm = np.memmap(path, np.uint8, "r")
    out = {}
    for name, shape, t, off in index:
        start = data_start + off
        if t == GGUF_F32:
            n = int(np.prod(shape))
            out[name] = jnp.asarray(np.frombuffer(mm[start : start + 4 * n], np.float32).reshape(shape), jnp.bfloat16)
        elif t == GGUF_Q4_0:
            rows, cols = shape
            blocks = mm[start : start + rows * cols // 32 * 18].reshape(rows, cols // 32, 18)
            if rows <= CHUNK_ROWS:
                out[name] = q4_0_expand_jit(np.ascontiguousarray(blocks))
            else:
                w = jnp.zeros(shape, jnp.bfloat16)
                for r0 in range(0, rows, CHUNK_ROWS):
                    w = q4_0_expand_into(w, np.ascontiguousarray(blocks[r0 : r0 + CHUNK_ROWS]), r0)
                out[name] = w
        else:
            raise TypeError(f"{name}: GGUF type {t} is neither F32 nor Q4_0")
    return jax.block_until_ready(out)


def engine_load(engine, tensors):
    """ENGINE.load, unchanged, fed `tensors` in place of the files it would read: it lists the folder's
    *.safetensors and reads each with safetensors.flax.load_file, so the folder holds one placeholder
    and that one call returns `tensors`."""
    folder = WORK / "engine-dir"
    folder.mkdir(exist_ok=True)
    for f in MODEL_DIR.iterdir():
        (folder / f.name).unlink(missing_ok=True)
        (folder / f.name).symlink_to(f.resolve())
    (folder / "gguf.safetensors").write_bytes(b"")
    real = safetensors.flax.load_file
    safetensors.flax.load_file = lambda p, *a, **k: tensors if p.endswith("gguf.safetensors") else real(p, *a, **k)
    try:
        engine.load(local_dir=str(folder))
    finally:
        safetensors.flax.load_file = real


TOK = ChatTokenizer(MODEL_DIR)
GEN_CFG = json.loads((MODEL_DIR / "generation_config.json").read_text())
EOS_IDS = GEN_CFG["eos_token_id"]
# Prompt plus answer. At 8,192 the prefill of a long prompt needs more HBM than this chip has left
# beside the unpacked weights (measured on a Colab v5e-1: 16.77 GB asked, 15.75 GB free), so the
# server stops at 4,096 and refuses longer prompts with a clear error.
MAX_MODEL_LEN = 4096
WARM_UP = [{"role": "user", "content": "Say hello."}]  # cell 2.2 sends this: prompt bucket 64, budget 512

# 1. Tensor shapes from the GGUF header: one 16 MB range request, while the file downloads.
t0 = time.perf_counter()
GGUF_URL = f"https://huggingface.co/{GGUF_REPO}/resolve/{GGUF_REV}/{GGUF_FILE}"
head = urllib.request.urlopen(urllib.request.Request(GGUF_URL, headers={"Range": "bytes=0-16777215"}), timeout=60)
DATA_START, GGUF_INDEX = gguf_index(io.BytesIO(head.read()))
print(f"GGUF header: {len(GGUF_INDEX)} tensors, read in {time.perf_counter() - t0:.1f} s")

# 2. The engine on zero weights of those shapes, and the warm-up request's programs compiled. XLA needs
#    only shapes and dtypes, so this runs during the download.
print(f"compiling the first request shape on zero weights, {time.perf_counter() - T_START:.0f} s after the first cell; "
      "this took 26 s on a Colab v5e-1 on 2026-11-20 ...", flush=True)  # fmt: skip
t0 = time.perf_counter()
ENGINE = JaxGemmaEngine(MODEL_ID, quant_mode="w4a16", max_model_len=MAX_MODEL_LEN, dequant_at_load=True)
engine_load(ENGINE, {name: jnp.zeros(shape, jnp.bfloat16) for name, shape, _, _ in GGUF_INDEX})
ENGINE.bos_token_id = TOK.bos_token_id
enc = TOK.apply_chat_template(WARM_UP, tokenize=True, add_generation_prompt=True, enable_thinking=False)
stream = ENGINE.generate_stream(list(enc["input_ids"] if hasattr(enc, "keys") else enc), 512, 0.0, -1, [], 0)
next(stream), next(stream)  # one prefill and one decode step: both programs compiled
stream.close()
ENGINE.params = None
STAGES["first shape compiled"] = time.perf_counter() - T_START
print(f"compiled in {time.perf_counter() - t0:.0f} s, {STAGES['first shape compiled']:.0f} s after the first cell")

# 3. The real weights, once the download is done, in place of the zeros. Same shapes and dtypes, so
#    the compiled programs are reused.
t0 = time.perf_counter()
if DOWNLOAD_THREAD.is_alive():
    print("waiting for the GGUF download (progress bar in cell 1.1) ...", flush=True)
DOWNLOAD_THREAD.join()
if "error" in DL:
    raise RuntimeError(f"download failed:\n{DL['error']}")
STAGES["download finished"] = DL["done_at"]
print(
    f"GGUF: {TOTAL_BYTES / 1e9:.2f} GB, downloaded in {DL['seconds']:.0f} s, done {STAGES['download finished']:.0f} s "
    f"after the first cell (this cell waited {time.perf_counter() - t0:.0f} s for it)"
)
t0 = time.perf_counter()
params = convert_safetensors_to_jax_params(
    gguf_to_chip(DL["path"], DATA_START, GGUF_INDEX),
    num_layers=ENGINE.config.num_hidden_layers,
    first_kv_shared_idx=ENGINE.config.first_kv_shared_layer_idx,
    attention_k_eq_v=ENGINE.config.attention_k_eq_v,
)
ENGINE.params = jax.device_put(params, ENGINE.device)
del params
ENGINE.weight_bytes = sum(int(x.size) * int(x.dtype.itemsize) for x in jax.tree_util.tree_leaves(ENGINE.params))
STAGES["weights on chip"] = time.perf_counter() - T_START
print(
    f"engine: tpu-jax {TPU_JAX_COMMIT[:7]}, {ENGINE.weight_bytes / 1e9:.2f} GB of bf16 weights on the chip, "
    f"expanded from the GGUF in {time.perf_counter() - t0:.0f} s"
)
print(f"HBM in use: {dev.memory_stats()['bytes_in_use'] / 1e9:.2f} of {HBM_LIMIT / 1e9:.2f} GB")

GGUF header: 540 tensors, read in 0.9 s
compiling the first request shape on zero weights, 12 s after the first cell; this took 26 s on a Colab v5e-1 on 2026-11-20 ...


compiled in 28 s, 40 s after the first cell
GGUF: 2.62 GB, downloaded in 27 s, done 29 s after the first cell (this cell waited 0 s for it)


engine: tpu-jax 4b9f8e9, 9.26 GB of bf16 weights on the chip, expanded from the GGUF in 8 s
HBM in use: 9.29 of 16.91 GB


In [ ]:
# Context experiment: one set of weights, several engine settings
import gc, traceback
from ports.gemma4.jax_e_model import make_cached_decode_step, prefill_with_kv_cache, Gemma4EModelJAX

BASE = ENGINE
def make_engine(max_len, chunk=None, window_kv=None):
    e = JaxGemmaEngine(MODEL_ID, quant_mode="w4a16", max_model_len=max_len, dequant_at_load=True,
                       prefill_chunk_size=chunk, window_kv=window_kv)
    e.config, e.params, e.device, e.model = BASE.config, BASE.params, BASE.device, BASE.model
    e.quant_mode, e.bos_token_id, e.weight_bytes = BASE.quant_mode, BASE.bos_token_id, BASE.weight_bytes
    if e.window_kv is None:
        e.window_kv = bool(e.config.sliding_window) and e.max_model_len > int(e.config.sliding_window)
    e._decode_step = jax.jit(make_cached_decode_step(e.model, quant_mode=e.quant_mode, window_kv=e.window_kv),
                             donate_argnums=(1, 2))
    e._jit_prefill = jax.jit(prefill_with_kv_cache,
                             static_argnames=("model", "max_new_tokens", "quant_mode", "cache_dtype", "window_kv"))
    return e

FACT = "The access code for the blue door is 7351."
FILL = [
    "The river bends twice before it reaches the old mill.",
    "Engineers measured the bridge again in the spring.",
    "A market opens every Tuesday near the station.",
    "The library keeps its maps on the third floor.",
    "Farmers planted barley along the eastern fields.",
]
def needle_prompt(n_tokens):
    """A chat prompt of exactly n_tokens: the fact first, filler, then the question."""
    q = "What is the access code for the blue door? Answer with the number only."
    body = [FACT]
    i = 0
    while True:
        msgs = [{"role": "user", "content": " ".join(body) + "\n\n" + q}]
        ids = TOK.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False)
        if len(ids) >= n_tokens:
            break
        body.append(FILL[i % len(FILL)]); i += 1
    while len(ids) > n_tokens:  # trim filler words from the end until it fits
        body[-1] = " ".join(body[-1].split()[:-1]) or FILL[0][:1]
        if not body[-1].strip():
            body.pop()
        msgs = [{"role": "user", "content": " ".join(body) + "\n\n" + q}]
        ids = TOK.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False)
    return ids

def run(e, ids, budget, n_out=64):
    t = time.perf_counter(); toks = []; t_first = None
    stream = e.generate_stream(ids, budget, 0.0, -1, EOS_IDS, 0)
    for item in stream:
        if not isinstance(item, int):
            break
        if t_first is None:
            t_first = time.perf_counter() - t
        toks.append(item)
        if len(toks) >= n_out:
            break
    stream.close()
    total = time.perf_counter() - t
    rate = (len(toks) - 1) / (total - t_first) if len(toks) > 1 and total > t_first else float("nan")
    return t_first, rate, TOK.decode(toks, skip_special_tokens=True).strip()

def hbm():
    m = dev.memory_stats(); return m["bytes_in_use"] / 1e9, m.get("peak_bytes_in_use", 0) / 1e9

ROWS = []
def case(label, max_len, chunk, window_kv, n_prompt, budget):
    e = make_engine(max_len, chunk, window_kv)
    ids = needle_prompt(n_prompt) if n_prompt > 200 else TOK.apply_chat_template(
        [{"role": "user", "content": "Describe the water cycle in about 150 words."}], tokenize=True,
        add_generation_prompt=True, enable_thinking=False)
    row = {"case": label, "max_len": max_len, "chunk": chunk, "window_kv": e.window_kv, "prompt": len(ids), "budget": budget}
    try:
        f1, r1, out = run(e, ids, budget)
        f2, r2, _ = run(e, ids, budget)  # same shape again: no compile if the programs are reused
        row.update(first_call_s=round(f1, 2), repeat_first_token_s=round(f2, 2), decode_tok_s=round(r2, 1),
                   answer=out[:60], ok=True)
    except Exception as ex:
        row.update(ok=False, error=f"{type(ex).__name__}: {str(ex).splitlines()[0][:160]}")
    row["hbm_in_use_gb"], row["hbm_peak_gb"] = [round(x, 2) for x in hbm()]
    ROWS.append(row); print(json.dumps(row), flush=True)
    del e; gc.collect()

print("weights", round(ENGINE.weight_bytes / 1e9, 2), "GB; HBM", [round(x, 2) for x in hbm()])
# decode speed at a short prompt, current settings vs full-cache settings
case("short, current (4096, one-pass)", 4096, None, None, 0, 2048)
case("short, 8192 chunked 512", 8192, 512, False, 0, 2048)
# long prompts
case("4096 one-pass, prompt 3830", 4096, None, None, 3830, 256)
case("8192 one-pass, prompt 7926", 8192, None, None, 7926, 256)
case("8192 chunked 512, prompt 7926", 8192, 512, False, 7926, 256)
case("16384 chunked 512, prompt 15800", 16384, 512, False, 15800, 256)
case("32768 chunked 512, prompt 32200", 32768, 512, False, 32200, 256)
json.dump(ROWS, open(WORK / "ctx_rows.json", "w"))


weights 9.26 GB; HBM [9.29, 13.04]


{"case": "short, current (4096, one-pass)", "max_len": 4096, "chunk": null, "window_kv": true, "prompt": 21, "budget": 2048, "first_call_s": 16.53, "repeat_first_token_s": 0.01, "decode_tok_s": 132.0, "answer": "The water cycle is the continuous movement of water on, abov", "ok": true, "hbm_in_use_gb": 9.31, "hbm_peak_gb": 13.04}


{"case": "short, 8192 chunked 512", "max_len": 8192, "chunk": 512, "window_kv": false, "prompt": 21, "budget": 2048, "first_call_s": 16.93, "repeat_first_token_s": 0.01, "decode_tok_s": 127.5, "answer": "The water cycle is the continuous movement of water on, abov", "ok": true, "hbm_in_use_gb": 9.32, "hbm_peak_gb": 13.04}


{"case": "4096 one-pass, prompt 3830", "max_len": 4096, "chunk": null, "window_kv": true, "prompt": 3830, "budget": 256, "first_call_s": 45.18, "repeat_first_token_s": 0.23, "decode_tok_s": 94.3, "answer": "7351", "ok": true, "hbm_in_use_gb": 9.31, "hbm_peak_gb": 13.04}


{"case": "8192 one-pass, prompt 7926", "max_len": 8192, "chunk": null, "window_kv": true, "prompt": 7926, "budget": 256, "ok": false, "error": "JaxRuntimeError: RESOURCE_EXHAUSTED: E1000: CompileTimeHbmOom: Ran out of memory on HBM, the total memory required for HLO temporaries (16.77G) exceeds available HBM (15.75G). H", "hbm_in_use_gb": 9.3, "hbm_peak_gb": 13.04}


{"case": "8192 chunked 512, prompt 7926", "max_len": 8192, "chunk": 512, "window_kv": false, "prompt": 7926, "budget": 256, "first_call_s": 79.6, "repeat_first_token_s": 1.9, "decode_tok_s": 86.5, "answer": "7351", "ok": true, "hbm_in_use_gb": 9.27, "hbm_peak_gb": 16.39}


{"case": "16384 chunked 512, prompt 15800", "max_len": 16384, "chunk": 512, "window_kv": false, "prompt": 15800, "budget": 256, "first_call_s": 79.88, "repeat_first_token_s": 3.19, "decode_tok_s": 83.1, "answer": "7351", "ok": true, "hbm_in_use_gb": 9.27, "hbm_peak_gb": 16.39}


{"case": "32768 chunked 512, prompt 32200", "max_len": 32768, "chunk": 512, "window_kv": false, "prompt": 32200, "budget": 256, "first_call_s": 84.33, "repeat_first_token_s": 7.59, "decode_tok_s": 74.0, "answer": "7351", "ok": true, "hbm_in_use_gb": 9.27, "hbm_peak_gb": 16.39}
